# Ogefmeeting — Test Deepgram STT (nouveau)

Notebook **propre** : transcription live micro → Deepgram (français).

- Pas d’asyncio
- Pas de websockets manuels
- SDK Deepgram synchrone uniquement

**Noyau requis :** `Python (ogefmeeting-IA)`

Fichier : `test-deepgram-stt-live.ipynb` (n’ouvre pas l’ancien notebook)

## Cellule 1 — Configuration

Exécute cette cellule en premier.

In [1]:
import os
import sys
from pathlib import Path

print("Python:", sys.executable)

from dotenv import load_dotenv

IA_DIR = Path.cwd()
if (IA_DIR / ".env").exists():
    load_dotenv(IA_DIR / ".env")
else:
    load_dotenv()

import sounddevice as sd
from deepgram import DeepgramClient, LiveOptions, LiveTranscriptionEvents

DEEPGRAM_API_KEY = os.getenv("DEEPGRAM_API_KEY", "").strip()
DEEPGRAM_LANGUAGE = os.getenv("DEEPGRAM_LANGUAGE", "fr").strip() or "fr"
DEEPGRAM_MODEL = os.getenv("DEEPGRAM_MODEL", "nova-3").strip() or "nova-3"
SAMPLE_RATE = 16000

assert DEEPGRAM_API_KEY, "DEEPGRAM_API_KEY manquant dans IA/.env"
assert ".venv" in sys.executable.replace("\\", "/").lower(), (
    "Mauvais noyau. Kernel → Change Kernel → Python (ogefmeeting-IA)"
)

print("OK — clé Deepgram:", DEEPGRAM_API_KEY[:6] + "…")
print("Modèle:", DEEPGRAM_MODEL, "| Langue:", DEEPGRAM_LANGUAGE)
print("Micro:", sd.query_devices(kind="input")["name"])

Python: C:\Users\DEBUZE DAVID\Documents\Ogefrem\ProjetReunion\Ogefmeeting\IA\.venv\Scripts\python.exe
OK — clé Deepgram: 296d54…
Modèle: nova-3 | Langue: fr
Micro: Microphone (Realtek(R) Audio)


## Cellule 2 — Transcription live

1. Exécute
2. **Entrée** pour démarrer
3. Parle en français
4. **Entrée** pour arrêter

In [ ]:
segments_finaux = []
erreurs = []


def on_open(self, open=None, **kwargs):
    print("Connecte a Deepgram")


def on_message(self, result=None, **kwargs):
    if result is None:
        return
    try:
        transcript = result.channel.alternatives[0].transcript
    except Exception:
        return
    if not transcript or not str(transcript).strip():
        return

    texte = str(transcript).strip()
    is_final = bool(getattr(result, "is_final", False))
    prefix = "[FINAL]" if is_final else "[Interim]"
    print(f"{prefix} {texte}")
    if is_final:
        segments_finaux.append(texte)


def on_error(self, error=None, **kwargs):
    msg = str(error)
    print("Erreur Deepgram:", msg)
    erreurs.append(msg)


def on_close(self, close=None, **kwargs):
    print("Connexion fermee")


client = DeepgramClient(DEEPGRAM_API_KEY)
dg = client.listen.live.v("1")

dg.on(LiveTranscriptionEvents.Open, on_open)
dg.on(LiveTranscriptionEvents.Transcript, on_message)
dg.on(LiveTranscriptionEvents.Error, on_error)
dg.on(LiveTranscriptionEvents.Close, on_close)

options = LiveOptions(
    model=DEEPGRAM_MODEL,
    language=DEEPGRAM_LANGUAGE,
    encoding="linear16",
    channels=1,
    sample_rate=SAMPLE_RATE,
    interim_results=True,
    punctuate=True,
    smart_format=True,
    endpointing=300,
)

print("Appuie sur Entree pour DEMARRER…")
input()

ok = dg.start(options)
if ok is False:
    raise RuntimeError("Echec demarrage Deepgram. Verifie DEEPGRAM_API_KEY dans .env")


def audio_callback(indata, frames, time_info, status):
    if status:
        print("Audio status:", status)
    dg.send(bytes(indata))


print("REC — parle en francais. Entree pour ARRETER.")
try:
    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="int16",
        blocksize=4096,
        callback=audio_callback,
    ):
        input()
finally:
    try:
        dg.finish()
    except Exception as e:
        print("finish():", e)

texte_complet = " ".join(segments_finaux).strip()
print("\n=== TEXTE FINAL ===\n")
print(texte_complet if texte_complet else "(vide)")
print(f"\n{len(segments_finaux)} segment(s) final(aux)")
if erreurs:
    print("Erreurs:", erreurs)

Appuie sur Entree pour DEMARRER…


C:\Users\DEBUZE DAVID\AppData\Local\Temp\ipykernel_28724\1726325635.py:38: DeprecatedWarning: live is deprecated as of 3.4.0 and will be removed in 4.0.0. deepgram.listen.live is deprecated. Use deepgram.listen.websocket instead.
  dg = client.listen.live.v("1")


Connecte a Deepgram
REC — parle en francais. Entree pour ARRETER.
[Interim] Vous
[Interim] Bonjour.
[Interim] Bonjour, je m'appelle
[FINAL] Bonjour, je m'appelle
[Interim] ingénieur David,
[Interim] ingénieur David,
[Interim] ingénieur David,
[Interim] ingénieur David.
[FINAL] ingénieur David.
[Interim] J'ai voulu
[Interim] Je voudrais intervenir
[Interim] Je voudrais intervenir dans le cadre de
[Interim] Je voudrais intervenir dans le cadre de ce que venait après
[FINAL] Je voudrais intervenir dans le cadre de ce que venait
[Interim] de dire mon ami,
[Interim] peine de dire mon ami, parce que j'ai
[Interim] peine de dire mon ami, parce que je ne suis pas d'accord
[Interim] peine de dire mon ami, parce que je ne suis pas d'accord de ce qu'il a
[Interim] peine de dire mon ami, parce que je ne suis pas d'accord de ce qu'il a avance comme hypothèse,
[Interim] peine de dire mon ami, parce que je ne suis pas d'accord de ce qu'il a avance comme hypothèse,
[Interim] peine de dire mon ami, par

ConnectionClosed in AbstractSyncWebSocketClient._listening with code 1006: 
send() failed - ConnectionClosed: no close frame received or sent
send() failed - ConnectionClosed: no close frame received or sent


Erreur Deepgram: {
    "description": "ConnectionClosed in AbstractSyncWebSocketClient._listening",
    "message": "no close frame received or sent",
    "type": "ConnectionClosed"
}
Connexion fermee


send() failed - ConnectionClosed: no close frame received or sent
